In [ ]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# =========================
# 1. Load datasets
# =========================
df_sample = pd.read_csv(
    '/content/drive/MyDrive/ LLM application for Maintenance Decision Making/Data/sample_df_500.csv'
)

df_2017 = pd.read_csv(
    '/content/drive/MyDrive/ LLM application for Maintenance Decision Making/Data/row-matching calculation/2017.csv'
)

# Clean column names
df_sample.columns = df_sample.columns.str.strip()
df_2017.columns = df_2017.columns.str.strip()

print("Sample shape:", df_sample.shape)
print("2017 shape:", df_2017.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sample shape: (500, 21)
2017 shape: (23554, 58)


In [ ]:
# =========================
# 2. Rename sample columns to match 2017 columns
# =========================
column_map = {
    'AADT': 'AADT_mean',
    'AADT Single Unit': 'AADT_Single_Unit_mean',
    'AADT Combination': 'AADT_Combination_mean',
    'Future AADT': 'Future_AADT_mean',
    'International Roughness Index': 'IRI_mean',
    'Thickness of Rigid Pavements': 'Thickness_Rigid_mean',
    'Thickness of Flexible Pavements': 'Thickness_Flexible_mean',
    'Thickness of Base Pavements': 'Base_Thickness_mean',
    'Functional System': 'F_System_mode',
    'Urban Type': 'Urban_Code_mode',
    'Surface Type': 'Surface_Type_mode',
    'Base Type': 'Base_Type_mode',
    'Rutting': 'Rutting_mean',
    'Cracking Percent': 'Cracking_Percent_mean',
    'Relative Humidity': 'RHU_AV',
    'Freezing Index': 'FRZ_IDX',
    'Average Temperature': 'TEMP_AVG',
    'Precipitation': 'PRECIPITATION'
}

df_sample_merge = df_sample.rename(columns=column_map)

merge_cols = list(column_map.values())

print("Merge columns:")
print(merge_cols)

Merge columns:
['AADT_mean', 'AADT_Single_Unit_mean', 'AADT_Combination_mean', 'Future_AADT_mean', 'IRI_mean', 'Thickness_Rigid_mean', 'Thickness_Flexible_mean', 'Base_Thickness_mean', 'F_System_mode', 'Urban_Code_mode', 'Surface_Type_mode', 'Base_Type_mode', 'Rutting_mean', 'Cracking_Percent_mean', 'RHU_AV', 'FRZ_IDX', 'TEMP_AVG', 'PRECIPITATION']


In [ ]:
# =========================
# 3. Keep only columns that really exist in both datasets
# =========================
merge_cols = [c for c in merge_cols if c in df_sample_merge.columns and c in df_2017.columns]

print("Final merge columns:", len(merge_cols))
print(merge_cols)

Final merge columns: 18
['AADT_mean', 'AADT_Single_Unit_mean', 'AADT_Combination_mean', 'Future_AADT_mean', 'IRI_mean', 'Thickness_Rigid_mean', 'Thickness_Flexible_mean', 'Base_Thickness_mean', 'F_System_mode', 'Urban_Code_mode', 'Surface_Type_mode', 'Base_Type_mode', 'Rutting_mean', 'Cracking_Percent_mean', 'RHU_AV', 'FRZ_IDX', 'TEMP_AVG', 'PRECIPITATION']


In [ ]:
# =========================
# 4. Convert numeric columns to numeric and round
# =========================
numeric_cols = [
    'AADT_mean',
    'AADT_Single_Unit_mean',
    'AADT_Combination_mean',
    'Future_AADT_mean',
    'IRI_mean',
    'Thickness_Rigid_mean',
    'Thickness_Flexible_mean',
    'Base_Thickness_mean',
    'Rutting_mean',
    'Cracking_Percent_mean',
    'RHU_AV',
    'FRZ_IDX',
    'TEMP_AVG',
    'PRECIPITATION'
]

numeric_cols = [c for c in numeric_cols if c in merge_cols]

for col in numeric_cols:
    df_sample_merge[col] = pd.to_numeric(df_sample_merge[col], errors='coerce').round(4)
    df_2017[col] = pd.to_numeric(df_2017[col], errors='coerce').round(4)

In [ ]:
# =========================
# 5. Convert categorical/code columns to string
# =========================
cat_cols = [
    'F_System_mode',
    'Urban_Code_mode',
    'Surface_Type_mode',
    'Base_Type_mode'
]

cat_cols = [c for c in cat_cols if c in merge_cols]

for col in cat_cols:
    df_sample_merge[col] = df_sample_merge[col].astype(str).str.strip()
    df_2017[col] = df_2017[col].astype(str).str.strip()

In [ ]:
# =========================
# 6. Check duplicate keys before merge
# =========================
print("Duplicate keys in sample:")
print(df_sample_merge.duplicated(subset=merge_cols).sum())

print("Duplicate keys in 2017:")
print(df_2017.duplicated(subset=merge_cols).sum())

Duplicate keys in sample:
7
Duplicate keys in 2017:
76


In [ ]:
# =========================
# 7. Merge county and state codes
# =========================
df_codes = df_2017[
    merge_cols + ['County_Code_mode', 'State_Code_mode']
].drop_duplicates(subset=merge_cols)

df_merged_temp = df_sample_merge.merge(
    df_codes,
    on=merge_cols,
    how='left'
)

print("Merged shape:", df_merged_temp.shape)
print("Missing County:", df_merged_temp['County_Code_mode'].isna().sum())
print("Missing State:", df_merged_temp['State_Code_mode'].isna().sum())

Merged shape: (500, 23)
Missing County: 500
Missing State: 500


In [ ]:
df_merged_temp

,AADT_mean,AADT_Single_Unit_mean,AADT_Combination_mean,Future_AADT_mean,IRI_mean,Thickness_Rigid_mean,Thickness_Flexible_mean,Base_Thickness_mean,F_System_mode,Urban_Code_mode,...,Cracking_Percent_mean,Faulting,RHU_AV,FRZ_IDX,TEMP_AVG,PRECIPITATION,Age,Maintenance Intervention,County_Code_mode,State_Code_mode
0,71446.0,4259.0,7445.0,85878.0,71.5,0.0,10.5,15.0,Interstate,rural,...,0.0,0.0,75.5,224.4,11.6,1318.4,58,Thin Overlay,NaN,NaN
1,8300.0,130.0,40.0,10300.0,108.0,0.0,12.0,17.0,Minor Arterial,urban,...,2.0,0.0,78.2,216.0,10.4,1435.3,26,No Maintenance,NaN,NaN
2,750.0,43.0,7.0,1000.0,242.5,0.0,3.0,12.0,Major Collector,urban,...,0.0,0.0,77.6,254.6,10.4,1398.8,9,Thin Overlay,NaN,NaN
3,4513.0,50.0,14.0,5032.0,236.0,0.0,10.0,6.0,Major Collector,urban,...,5.5,0.0,75.3,227.2,12.2,1246.0,67,No Maintenance,NaN,NaN
4,8875.0,502.0,28.0,13188.0,116.3,0.0,6.5,12.0,Minor Arterial,urban,...,0.3,0.0,84.0,25.0,9.4,1628.8,43,Thin Overlay,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,24743.0,1039.0,4478.0,26820.0,58.0,0.0,9.0,28.0,Interstate,rural,...,0.0,0.0,81.2,622.0,7.4,1444.6,16,Thin Overlay,NaN,NaN
496,4200.0,874.0,986.0,6461.0,99.9,0.0,2.0,8.0,Minor Arterial,rural,...,8.0,0.0,76.2,0.0,22.1,1129.0,9,Thin Overlay,NaN,NaN
497,28700.0,1318.0,754.0,28800.0,120.0,7.0,1.0,10.0,Principal Arterial – Other,urban,...,2.0,0.0,78.2,130.1,11.5,1379.0,8,No Maintenance,NaN,NaN
498,63307.0,3119.0,7145.0,76728.0,72.0,0.0,9.5,18.0,Interstate,urban,...,0.0,0.0,77.7,393.9,9.3,1382.1,59,Thin Overlay,NaN,NaN


In [ ]:
county_list = []
state_list = []

for _, row in df_sample.iterrows():

    matches = df_2017[
        (df_2017['AADT_mean'].round(2) == round(row['AADT'], 2)) &
        (df_2017['IRI_mean'].round(2) == round(row['International Roughness Index'], 2)) &
        (df_2017['Rutting_mean'].round(2) == round(row['Rutting'], 2))
    ]

    if len(matches) == 1:
        county_list.append(matches.iloc[0]['County_Code_mode'])
        state_list.append(matches.iloc[0]['State_Code_mode'])

    elif len(matches) > 1:
        # multiple matches found
        county_list.append(matches.iloc[0]['County_Code_mode'])
        state_list.append(matches.iloc[0]['State_Code_mode'])

    else:
        county_list.append(None)
        state_list.append(None)

df_sample['County_Code_mode'] = county_list
df_sample['State_Code_mode'] = state_list

print("Matched rows:", df_sample['County_Code_mode'].notna().sum())
print("Missing rows:", df_sample['County_Code_mode'].isna().sum())

Matched rows: 482
Missing rows: 18


In [ ]:
county_list = []
state_list = []

for _, row in df_sample.iterrows():

    matches = df_2017[
        (df_2017['AADT_mean'].round(2) == round(row['AADT'], 2)) &
        (df_2017['IRI_mean'].round(2) == round(row['International Roughness Index'], 2)) &
        (df_2017['Rutting_mean'].round(2) == round(row['Rutting'], 2)) &
        (df_2017['Cracking_Percent_mean'].round(2) == round(row['Cracking Percent'], 2))
    ]

    if len(matches) > 0:
        county_list.append(matches.iloc[0]['County_Code_mode'])
        state_list.append(matches.iloc[0]['State_Code_mode'])
    else:
        county_list.append(None)
        state_list.append(None)

df_sample['County_Code_mode'] = county_list
df_sample['State_Code_mode'] = state_list

print("Matched rows:", df_sample['County_Code_mode'].notna().sum())
print("Missing rows:", df_sample['County_Code_mode'].isna().sum())

Matched rows: 482
Missing rows: 18


In [ ]:
dup_rows = df_sample.duplicated(
    subset=['AADT', 'International Roughness Index', 'Rutting'],
    keep=False
)

print("Number of duplicate rows:", dup_rows.sum())

df_sample[dup_rows].sort_values(
    ['AADT', 'International Roughness Index', 'Rutting']
)

Number of duplicate rows: 14


,AADT,AADT Single Unit,AADT Combination,Future AADT,International Roughness Index,Thickness of Rigid Pavements,Thickness of Flexible Pavements,Thickness of Base Pavements,Functional System,Urban Type,...,Cracking Percent,Faulting,Relative Humidity,Freezing Index,Average Temperature,Precipitation,Age,Maintenance Intervention,County_Code_mode,State_Code_mode
275,2900.0,820.0,502.0,6300.0,106.0,0.0,9.0,16.0,Principal Arterial – Other,small urban,...,0.0,0.0,78.9,489.4,8.6,1569.7,79,No Maintenance,5.0,9.0
297,2900.0,820.0,502.0,6300.0,106.0,0.0,9.0,16.0,Principal Arterial – Other,small urban,...,0.0,0.0,78.9,489.4,8.6,1569.7,79,No Maintenance,5.0,9.0
162,4684.0,234.0,116.0,5483.0,60.0,0.0,4.5,6.0,Principal Arterial – Other,rural,...,0.0,0.0,77.6,459.1,8.7,1412.2,45,No Maintenance,27.0,42.0
288,4684.0,234.0,116.0,5483.0,60.0,0.0,4.5,6.0,Principal Arterial – Other,rural,...,0.0,0.0,77.6,459.1,8.7,1412.2,45,No Maintenance,27.0,42.0
332,5031.0,249.0,62.0,5433.0,141.7,8.0,3.0,8.0,Minor Arterial,rural,...,5.7,0.0,78.8,44.6,13.3,1281.0,88,No Maintenance,9.0,34.0
472,5031.0,249.0,62.0,5433.0,141.7,8.0,3.0,8.0,Minor Arterial,rural,...,5.7,0.0,78.8,44.6,13.3,1281.0,88,No Maintenance,9.0,34.0
25,7261.0,392.0,80.0,9217.0,204.0,0.0,3.0,24.0,Minor Arterial,rural,...,31.0,0.0,80.2,695.0,7.0,1299.6,64,Thin Overlay,5.0,33.0
73,7261.0,392.0,80.0,9217.0,204.0,0.0,3.0,24.0,Minor Arterial,rural,...,31.0,0.0,80.2,695.0,7.0,1299.6,64,Thin Overlay,5.0,33.0
176,27480.0,2438.0,4178.0,33358.0,68.0,10.0,4.5,18.0,Interstate,rural,...,0.0,0.0,77.6,395.4,9.5,1362.7,60,Thin Overlay,19.0,42.0
195,27480.0,2438.0,4178.0,33358.0,68.0,10.0,4.5,18.0,Interstate,rural,...,0.0,0.0,77.6,395.4,9.5,1362.7,60,Thin Overlay,19.0,42.0


In [ ]:
dup_2017 = df_2017.duplicated(
    subset=['AADT_mean', 'IRI_mean', 'Rutting_mean'],
    keep=False
)

print("Number of duplicate rows:", dup_2017.sum())

Number of duplicate rows: 355


In [ ]:
dup_sample = df_sample.duplicated(
    subset=[
        'AADT',
        'International Roughness Index',
        'Rutting',
        'Cracking Percent',
        'Future AADT'
    ],
    keep=False
)

print("Sample duplicate rows:", dup_sample.sum())

Sample duplicate rows: 14


In [ ]:
dup_sample = df_sample[
    df_sample.duplicated(
        subset=['AADT', 'International Roughness Index', 'Rutting'],
        keep=False
    )
]

print(dup_sample.shape)
dup_sample

(14, 23)


,AADT,AADT Single Unit,AADT Combination,Future AADT,International Roughness Index,Thickness of Rigid Pavements,Thickness of Flexible Pavements,Thickness of Base Pavements,Functional System,Urban Type,...,Cracking Percent,Faulting,Relative Humidity,Freezing Index,Average Temperature,Precipitation,Age,Maintenance Intervention,County_Code_mode,State_Code_mode
7,47262.0,792.0,2978.0,51875.0,55.0,10.0,4.0,16.0,Interstate,urban,...,0.0,0.0,76.3,330.1,10.3,1280.2,55,No Maintenance,3.0,42.0
25,7261.0,392.0,80.0,9217.0,204.0,0.0,3.0,24.0,Minor Arterial,rural,...,31.0,0.0,80.2,695.0,7.0,1299.6,64,Thin Overlay,5.0,33.0
59,63307.0,3119.0,7145.0,76728.0,72.0,0.0,9.5,18.0,Interstate,urban,...,0.0,0.0,77.7,393.9,9.3,1382.1,59,Thin Overlay,79.0,42.0
73,7261.0,392.0,80.0,9217.0,204.0,0.0,3.0,24.0,Minor Arterial,rural,...,31.0,0.0,80.2,695.0,7.0,1299.6,64,Thin Overlay,5.0,33.0
162,4684.0,234.0,116.0,5483.0,60.0,0.0,4.5,6.0,Principal Arterial – Other,rural,...,0.0,0.0,77.6,459.1,8.7,1412.2,45,No Maintenance,27.0,42.0
176,27480.0,2438.0,4178.0,33358.0,68.0,10.0,4.5,18.0,Interstate,rural,...,0.0,0.0,77.6,395.4,9.5,1362.7,60,Thin Overlay,19.0,42.0
195,27480.0,2438.0,4178.0,33358.0,68.0,10.0,4.5,18.0,Interstate,rural,...,0.0,0.0,77.6,395.4,9.5,1362.7,60,Thin Overlay,19.0,42.0
275,2900.0,820.0,502.0,6300.0,106.0,0.0,9.0,16.0,Principal Arterial – Other,small urban,...,0.0,0.0,78.9,489.4,8.6,1569.7,79,No Maintenance,5.0,9.0
288,4684.0,234.0,116.0,5483.0,60.0,0.0,4.5,6.0,Principal Arterial – Other,rural,...,0.0,0.0,77.6,459.1,8.7,1412.2,45,No Maintenance,27.0,42.0
297,2900.0,820.0,502.0,6300.0,106.0,0.0,9.0,16.0,Principal Arterial – Other,small urban,...,0.0,0.0,78.9,489.4,8.6,1569.7,79,No Maintenance,5.0,9.0


In [ ]:
for idx, row in dup_sample.iterrows():

    matches = df_2017[
        (df_2017['AADT_mean'] == row['AADT']) &
        (df_2017['IRI_mean'] == row['International Roughness Index']) &
        (df_2017['Rutting_mean'] == row['Rutting'])
    ]

    print(f"Sample row {idx}: {len(matches)} matches found")

Sample row 7: 2 matches found
Sample row 25: 1 matches found
Sample row 59: 1 matches found
Sample row 73: 1 matches found
Sample row 162: 1 matches found
Sample row 176: 1 matches found
Sample row 195: 1 matches found
Sample row 275: 3 matches found
Sample row 288: 1 matches found
Sample row 297: 3 matches found
Sample row 332: 1 matches found
Sample row 403: 2 matches found
Sample row 472: 1 matches found
Sample row 498: 1 matches found


In [ ]:
column_map_10 = {
    'AADT': 'AADT_mean',
    'AADT Single Unit': 'AADT_Single_Unit_mean',
    'AADT Combination': 'AADT_Combination_mean',
    'Future AADT': 'Future_AADT_mean',
    'International Roughness Index': 'IRI_mean',
    'Rutting': 'Rutting_mean',
    'Cracking Percent': 'Cracking_Percent_mean',
    'Thickness of Rigid Pavements': 'Thickness_Rigid_mean',
    'Thickness of Flexible Pavements': 'Thickness_Flexible_mean',
    'Thickness of Base Pavements': 'Base_Thickness_mean'
}

df_sample_merge = df_sample.rename(columns=column_map_10)

merge_cols = list(column_map_10.values())

for col in merge_cols:
    df_sample_merge[col] = pd.to_numeric(df_sample_merge[col], errors='coerce').round(4)
    df_2017[col] = pd.to_numeric(df_2017[col], errors='coerce').round(4)

df_codes = df_2017[
    merge_cols + ['County_Code_mode', 'State_Code_mode']
].drop_duplicates(subset=merge_cols)

df_merged_temp = df_sample_merge.merge(
    df_codes,
    on=merge_cols,
    how='left'
)

df_final = df_sample.copy()
df_final['County_Code_mode'] = df_merged_temp['County_Code_mode']
df_final['State_Code_mode'] = df_merged_temp['State_Code_mode']

print("Matched rows:", df_final['County_Code_mode'].notna().sum())
print("Missing rows:", df_final['County_Code_mode'].isna().sum())

df_final.to_csv(
    '/content/drive/MyDrive/ LLM application for Maintenance Decision Making/Data/sample_df_500_with_state_county.csv',
    index=False
)

KeyError: 'County_Code_mode'

In [ ]:
county_list = []
state_list = []

for _, row in df_sample.iterrows():

    matches = df_2017[
        (df_2017['AADT_mean'].round(4) == round(row['AADT'], 4)) &
        (df_2017['AADT_Single_Unit_mean'].round(4) == round(row['AADT Single Unit'], 4)) &
        (df_2017['AADT_Combination_mean'].round(4) == round(row['AADT Combination'], 4)) &
        (df_2017['Future_AADT_mean'].round(4) == round(row['Future AADT'], 4)) &
        (df_2017['IRI_mean'].round(4) == round(row['International Roughness Index'], 4)) &
        (df_2017['Rutting_mean'].round(4) == round(row['Rutting'], 4)) &
        (df_2017['Cracking_Percent_mean'].round(4) == round(row['Cracking Percent'], 4))
    ]

    if len(matches) > 0:
        county_list.append(matches.iloc[0]['County_Code_mode'])
        state_list.append(matches.iloc[0]['State_Code_mode'])
    else:
        county_list.append(None)
        state_list.append(None)

df_sample['County_Code_mode'] = county_list
df_sample['State_Code_mode'] = state_list

print("Matched rows:", df_sample['County_Code_mode'].notna().sum())
print("Missing rows:", df_sample['County_Code_mode'].isna().sum())

Matched rows: 482
Missing rows: 18


In [ ]:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Load datasets
df_sample = pd.read_csv(
    '/content/drive/MyDrive/ LLM application for Maintenance Decision Making/Data/sample_df_500.csv'
)

df_2017 = pd.read_csv(
    '/content/drive/MyDrive/ LLM application for Maintenance Decision Making/Data/row-matching calculation/2017.csv'
)

# Clean column names
df_sample.columns = df_sample.columns.str.strip()
df_2017.columns = df_2017.columns.str.strip()

# Convert matching columns to numeric
sample_cols = [
    'AADT',
    'AADT Single Unit',
    'AADT Combination',
    'Future AADT',
    'International Roughness Index',
    'Rutting',
    'Cracking Percent'
]

original_cols = [
    'AADT_mean',
    'AADT_Single_Unit_mean',
    'AADT_Combination_mean',
    'Future_AADT_mean',
    'IRI_mean',
    'Rutting_mean',
    'Cracking_Percent_mean'
]

for s_col, o_col in zip(sample_cols, original_cols):
    df_sample[s_col] = pd.to_numeric(df_sample[s_col], errors='coerce')
    df_2017[o_col] = pd.to_numeric(df_2017[o_col], errors='coerce')

# Row-by-row matching
county_list = []
state_list = []

for _, row in df_sample.iterrows():

    matches = df_2017[
        (df_2017['AADT_mean'].round(4) == round(row['AADT'], 4)) &
        (df_2017['AADT_Single_Unit_mean'].round(4) == round(row['AADT Single Unit'], 4)) &
        (df_2017['AADT_Combination_mean'].round(4) == round(row['AADT Combination'], 4)) &
        (df_2017['Future_AADT_mean'].round(4) == round(row['Future AADT'], 4)) &
        (df_2017['IRI_mean'].round(4) == round(row['International Roughness Index'], 4)) &
        (df_2017['Rutting_mean'].round(4) == round(row['Rutting'], 4)) &
        (df_2017['Cracking_Percent_mean'].round(4) == round(row['Cracking Percent'], 4))
    ]

    if len(matches) > 0:
        county_list.append(matches.iloc[0]['County_Code_mode'])
        state_list.append(matches.iloc[0]['State_Code_mode'])
    else:
        county_list.append(None)
        state_list.append(None)

# Add FIPS code columns
df_sample['County_Code_mode'] = county_list
df_sample['State_Code_mode'] = state_list

print("Matched rows:", df_sample['County_Code_mode'].notna().sum())
print("Missing rows:", df_sample['County_Code_mode'].isna().sum())

# Extract matched and unmatched rows
df_matched = df_sample[df_sample['County_Code_mode'].notna()].copy()
df_unmatched = df_sample[df_sample['County_Code_mode'].isna()].copy()

print("Matched shape:", df_matched.shape)
print("Unmatched shape:", df_unmatched.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Matched rows: 482
Missing rows: 18
Matched shape: (482, 23)
Unmatched shape: (18, 23)


In [ ]:
df_matched

,AADT,AADT Single Unit,AADT Combination,Future AADT,International Roughness Index,Thickness of Rigid Pavements,Thickness of Flexible Pavements,Thickness of Base Pavements,Functional System,Urban Type,...,Cracking Percent,Faulting,Relative Humidity,Freezing Index,Average Temperature,Precipitation,Age,Maintenance Intervention,County_Code_mode,State_Code_mode
0,71446.0,4259.0,7445.0,85878.0,71.5,0.0,10.5,15.0,Interstate,rural,...,0.0,0.0,75.5,224.4,11.6,1318.4,58,Thin Overlay,21.0,24.0
1,8300.0,130.0,40.0,10300.0,108.0,0.0,12.0,17.0,Minor Arterial,urban,...,2.0,0.0,78.2,216.0,10.4,1435.3,26,No Maintenance,11.0,9.0
2,750.0,43.0,7.0,1000.0,242.5,0.0,3.0,12.0,Major Collector,urban,...,0.0,0.0,77.6,254.6,10.4,1398.8,9,Thin Overlay,7.0,9.0
3,4513.0,50.0,14.0,5032.0,236.0,0.0,10.0,6.0,Major Collector,urban,...,5.5,0.0,75.3,227.2,12.2,1246.0,67,No Maintenance,61.0,39.0
4,8875.0,502.0,28.0,13188.0,116.3,0.0,6.5,12.0,Minor Arterial,urban,...,0.3,0.0,84.0,25.0,9.4,1628.8,43,Thin Overlay,53.0,53.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,24743.0,1039.0,4478.0,26820.0,58.0,0.0,9.0,28.0,Interstate,rural,...,0.0,0.0,81.2,622.0,7.4,1444.6,16,Thin Overlay,23.0,23.0
496,4200.0,874.0,986.0,6461.0,99.9,0.0,2.0,8.0,Minor Arterial,rural,...,8.0,0.0,76.2,0.0,22.1,1129.0,9,Thin Overlay,105.0,12.0
497,28700.0,1318.0,754.0,28800.0,120.0,7.0,1.0,10.0,Principal Arterial – Other,urban,...,2.0,0.0,78.2,130.1,11.5,1379.0,8,No Maintenance,1.0,9.0
498,63307.0,3119.0,7145.0,76728.0,72.0,0.0,9.5,18.0,Interstate,urban,...,0.0,0.0,77.7,393.9,9.3,1382.1,59,Thin Overlay,79.0,42.0


In [ ]:
df_unmatched

,AADT,AADT Single Unit,AADT Combination,Future AADT,International Roughness Index,Thickness of Rigid Pavements,Thickness of Flexible Pavements,Thickness of Base Pavements,Functional System,Urban Type,...,Cracking Percent,Faulting,Relative Humidity,Freezing Index,Average Temperature,Precipitation,Age,Maintenance Intervention,County_Code_mode,State_Code_mode
12,1896.0,56.0,66.0,2275.0,92.0,8.0,0.0,6.0,Major Collector,rural,...,0.0,0.0,69.4,1464.8,4.6,675.9,63,No Maintenance,NaN,NaN
27,81000.0,3160.0,9160.0,86900.0,92.0,14.0,0.0,11.0,Interstate,urban,...,0.0,0.0,74.2,49.8,14.6,1219.1,12,Resurfacing,NaN,NaN
46,29514.0,2019.0,5942.0,40546.0,157.0,12.0,0.0,8.0,Interstate,rural,...,0.0,0.0,77.6,352.9,9.7,1347.0,26,No Maintenance,NaN,NaN
51,126000.0,3320.0,2060.0,155100.0,67.3,11.0,0.0,10.0,Principal Arterial – Other Freeways and Expres...,urban,...,0.0,0.0,76.1,28.6,15.6,1328.6,19,Resurfacing,NaN,NaN
93,21647.0,455.0,5196.0,21703.0,58.0,12.0,0.0,18.0,Interstate,small urban,...,0.0,0.0,78.6,272.6,9.4,1097.5,8,No Maintenance,NaN,NaN
159,49713.0,1619.0,8225.0,59347.0,165.5,13.0,0.0,8.0,Interstate,urban,...,0.0,0.0,79.0,516.1,8.1,1394.0,28,Thick Overlay,NaN,NaN
200,3908.0,348.0,370.0,4250.0,61.5,9.0,0.0,4.0,Principal Arterial – Other Freeways and Expres...,small urban,...,0.0,0.0,70.9,957.0,7.2,999.2,24,Resurfacing,NaN,NaN
211,71700.0,1793.0,3083.0,102531.0,58.5,12.0,0.0,10.0,Interstate,urban,...,0.0,0.0,74.8,41.0,14.4,1427.3,18,Thin Overlay,NaN,NaN
254,43546.0,2362.0,9610.0,52778.0,116.0,13.0,0.0,12.0,Interstate,urban,...,0.0,0.0,77.7,393.9,9.3,1382.1,28,Thick Overlay,NaN,NaN
255,17452.0,839.0,2909.0,20873.0,94.5,12.0,0.0,9.0,Principal Arterial – Other,rural,...,0.0,0.0,76.2,326.2,10.1,1295.6,21,No Maintenance,NaN,NaN


In [ ]:
results = []

for idx, row in df_unmatched.iterrows():

    matches = df_2017[
        (df_2017['AADT_mean'].round(4) == round(row['AADT'], 4)) &
        (df_2017['AADT_Single_Unit_mean'].round(4) == round(row['AADT Single Unit'], 4)) &
        (df_2017['AADT_Combination_mean'].round(4) == round(row['AADT Combination'], 4)) &
        (df_2017['Future_AADT_mean'].round(4) == round(row['Future AADT'], 4)) &
        (df_2017['IRI_mean'].round(4) == round(row['International Roughness Index'], 4)) &
        (df_2017['Rutting_mean'].round(4) == round(row['Rutting'], 4)) &
        (df_2017['Cracking_Percent_mean'].round(4) == round(row['Cracking Percent'], 4))
    ]

    results.append({
        'Sample_Index': idx,
        'Matches_in_2017': len(matches),
        'County_Code_mode': matches.iloc[0]['County_Code_mode'] if len(matches) > 0 else None,
        'State_Code_mode': matches.iloc[0]['State_Code_mode'] if len(matches) > 0 else None
    })

check_unmatched = pd.DataFrame(results)

check_unmatched

,Sample_Index,Matches_in_2017,County_Code_mode,State_Code_mode
0,12,0,None,None
1,27,0,None,None
2,46,0,None,None
3,51,0,None,None
4,93,0,None,None
5,159,0,None,None
6,200,0,None,None
7,211,0,None,None
8,254,0,None,None
9,255,0,None,None


In [ ]:
row = df_unmatched.iloc[0]

print(row[
    ['AADT',
     'AADT Single Unit',
     'AADT Combination',
     'Future AADT',
     'International Roughness Index',
     'Rutting',
     'Cracking Percent']
])

AADT                             1896.0
AADT Single Unit                   56.0
AADT Combination                   66.0
Future AADT                      2275.0
International Roughness Index      92.0
Rutting                             0.0
Cracking Percent                    0.0
Name: 12, dtype: object


In [ ]:
df_2017[df_2017['AADT_mean'] == row['AADT']].shape

(2, 58)

In [ ]:
df_2017[
    (df_2017['AADT_mean'] == row['AADT']) &
    (df_2017['IRI_mean'] == row['International Roughness Index'])
].shape

(1, 58)

In [ ]:
row = df_unmatched.iloc[0]

m1 = df_2017[df_2017['AADT_mean'] == row['AADT']]
print("AADT:", m1.shape)

m2 = m1[m1['IRI_mean'] == row['International Roughness Index']]
print("AADT + IRI:", m2.shape)

m3 = m2[m2['Rutting_mean'] == row['Rutting']]
print("AADT + IRI + Rutting:", m3.shape)

m4 = m3[m3['Cracking_Percent_mean'] == row['Cracking Percent']]
print("AADT + IRI + Rutting + Cracking:", m4.shape)

m5 = m4[m4['AADT_Single_Unit_mean'] == row['AADT Single Unit']]
print("+ AADT Single Unit:", m5.shape)

m6 = m5[m5['AADT_Combination_mean'] == row['AADT Combination']]
print("+ AADT Combination:", m6.shape)

m7 = m6[m6['Future_AADT_mean'] == row['Future AADT']]
print("+ Future AADT:", m7.shape)

AADT: (2, 58)
AADT + IRI: (1, 58)
AADT + IRI + Rutting: (0, 58)
AADT + IRI + Rutting + Cracking: (0, 58)
+ AADT Single Unit: (0, 58)
+ AADT Combination: (0, 58)
+ Future AADT: (0, 58)


In [ ]:
# Count duplicate AADT values in unmatched rows
dup_aadt = df_unmatched[
    df_unmatched.duplicated(subset=['AADT'], keep=False)
]

print("Rows with duplicate AADT:", len(dup_aadt))

dup_aadt.sort_values('AADT')

Rows with duplicate AADT: 0


,AADT,AADT Single Unit,AADT Combination,Future AADT,International Roughness Index,Thickness of Rigid Pavements,Thickness of Flexible Pavements,Thickness of Base Pavements,Functional System,Urban Type,...,Cracking Percent,Faulting,Relative Humidity,Freezing Index,Average Temperature,Precipitation,Age,Maintenance Intervention,County_Code_mode,State_Code_mode


In [ ]:
aadt_counts = df_unmatched['AADT'].value_counts()

print(aadt_counts[aadt_counts > 1])

Series([], Name: count, dtype: int64)


In [ ]:
aadt_results = []

for idx, row in df_unmatched.iterrows():

    matches = df_2017[
        df_2017['AADT_mean'] == row['AADT']
    ]

    aadt_results.append({
        'Sample_Index': idx,
        'AADT': row['AADT'],
        'Matches_Found': len(matches),
        'County_Code_mode': matches.iloc[0]['County_Code_mode'] if len(matches) > 0 else None,
        'State_Code_mode': matches.iloc[0]['State_Code_mode'] if len(matches) > 0 else None
    })

aadt_search_results = pd.DataFrame(aadt_results)

aadt_search_results

,Sample_Index,AADT,Matches_Found,County_Code_mode,State_Code_mode
0,12,1896.0,2,53.0,53.0
1,27,81000.0,7,57.0,37.0
2,46,29514.0,11,89.0,42.0
3,51,126000.0,4,63.0,37.0
4,93,21647.0,1,159.0,26.0
5,159,49713.0,3,69.0,42.0
6,200,3908.0,1,67.0,27.0
7,211,71700.0,1,45.0,45.0
8,254,43546.0,9,79.0,42.0
9,255,17452.0,1,99.0,42.0


In [89]:
# ======================================================
# Install required packages
# ======================================================
!pip install geopandas cartopy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 91.8 MB/s eta 0:00:00


In [88]:
fips_to_name = {
    24:'Maryland', 9:'Connecticut', 39:'Ohio', 53:'Washington',
    48:'Texas', 42:'Pennsylvania', 37:'North Carolina',
    33:'New Hampshire', 12:'Florida', 41:'Oregon',
    23:'Maine', 17:'Illinois', 44:'Rhode Island',
    30:'Montana', 32:'Nevada', 38:'North Dakota',
    51:'Virginia', 34:'New Jersey', 45:'South Carolina',
    27:'Minnesota', 26:'Michigan', 5:'Arkansas',
    20:'Kansas', 46:'South Dakota', 56:'Wyoming',
    55:'Wisconsin'
}

map_df = pd.DataFrame({
    'State': [fips_to_abbr[x] for x in selected_fips],
    'State_Name': [fips_to_name[x] for x in selected_fips],
    'Value': 1
})

import plotly.express as px

fig = px.choropleth(
    map_df,
    locations='State',
    locationmode='USA-states',
    color='Value',
    hover_name='State_Name',
    scope='usa',
    color_continuous_scale=['#4F81BD', '#4F81BD']
)

fig.update_layout(
    coloraxis_showscale=False,
    title='States Included in Pavement Dataset'
)

fig.show()

NameError: name 'selected_fips' is not defined